In [91]:
# ============================================================
# DocuMind - Agentic Orchestration
# Cell 1: Basic Setup
# ============================================================

import sys
import os
import re
import json
import torch
import numpy as np
import pandas as pd

from pathlib import Path

print("DocuMind Agentic Orchestration")
print("=" * 60)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DocuMind Agentic Orchestration
Python: 3.12.10
PyTorch: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [92]:
# ============================================================
# Cell 2: Load Retrieval Data
# ============================================================

import bm25s

processed_dir = Path("../processed")

# Full text chunks
full_chunks_df = pd.read_parquet(
    processed_dir / "full_text_chunks.parquet"
)

# Saved BGE embeddings
full_embeddings = np.load(
    processed_dir / "full_text_embeddings.npy",
    mmap_mode="r"
)

# Saved BM25 index
full_bm25s = bm25s.BM25.load(
    str(processed_dir / "full_text_bm25s"),
    load_corpus=False
)

print("Retrieval data loaded")
print("-" * 50)
print("Chunks:", full_chunks_df.shape)
print("Embeddings:", full_embeddings.shape)
print("Embedding dtype:", full_embeddings.dtype)
print("BM25:", type(full_bm25s).__name__)

Retrieval data loaded
--------------------------------------------------
Chunks: (223234, 6)
Embeddings: (223234, 384)
Embedding dtype: float32
BM25: BM25


In [93]:
# ============================================================
# Cell 3: Load Semantic Embedding Model
# ============================================================

from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device=device
)

print("Embedding model loaded")
print("-" * 50)
print("Model: BAAI/bge-small-en-v1.5")
print("Device:", device)
print("Dimension:", embedding_model.get_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded
--------------------------------------------------
Model: BAAI/bge-small-en-v1.5
Device: cuda
Dimension: 384


In [94]:
# ============================================================
# Cell 4: Load Qwen
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM

qwen_name = "Qwen/Qwen2.5-1.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_name)

qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

qwen_model.eval()

print("Qwen loaded")
print("Device:", qwen_model.device)
print(
    "GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


Qwen loaded
Device: cpu
GPU memory: 5.14 GB


In [95]:
print("Qwen device map:")
print(qwen_model.hf_device_map)

print("\nGPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("GPU memory reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

Qwen device map:
{'model.embed_tokens': 'cpu', 'lm_head': 'cpu', 'model.layers.0': 'cpu', 'model.layers.1': 'disk', 'model.layers.2': 'disk', 'model.layers.3': 'disk', 'model.layers.4': 'disk', 'model.layers.5': 'disk', 'model.layers.6': 'disk', 'model.layers.7': 'disk', 'model.layers.8': 'disk', 'model.layers.9': 'disk', 'model.layers.10': 'disk', 'model.layers.11': 'disk', 'model.layers.12': 'disk', 'model.layers.13': 'disk', 'model.layers.14': 'disk', 'model.layers.15': 'disk', 'model.layers.16': 'disk', 'model.layers.17': 'disk', 'model.layers.18': 'disk', 'model.layers.19': 'disk', 'model.layers.20': 'disk', 'model.layers.21': 'disk', 'model.layers.22': 'disk', 'model.layers.23': 'disk', 'model.layers.24': 'disk', 'model.layers.25': 'disk', 'model.layers.26': 'disk', 'model.layers.27': 'disk', 'model.norm': 'disk', 'model.rotary_emb': 'disk'}

GPU memory allocated: 5.14 GB
GPU memory reserved: 5.24 GB


In [96]:
import importlib.util

print("bitsandbytes installed:",
      importlib.util.find_spec("bitsandbytes") is not None)

bitsandbytes installed: False


In [97]:
import gc
import torch

# Remove the current CPU/disk-offloaded Qwen
del qwen_model
del qwen_tokenizer

# Move BGE off the GPU temporarily
embedding_model.to("cpu")

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU memory before reload:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

GPU memory before reload: 4.89 GB


In [98]:
import gc
import torch

total = 0
objects = []

for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) and obj.is_cuda:
            size_gb = obj.numel() * obj.element_size() / (1024**3)
            if size_gb > 0.01:
                objects.append((size_gb, str(obj.dtype), tuple(obj.shape)))
                total += size_gb
    except:
        pass

objects.sort(reverse=True)

print("Live CUDA tensors found:", len(objects))
print("Approximate tensor memory:", round(total, 2), "GB")

for item in objects[:10]:
    print(item)

Live CUDA tensors found: 122
Approximate tensor memory: 4.09 GB
(0.4346923828125, 'torch.float16', (151936, 1536))
(0.4346923828125, 'torch.float16', (151936, 1536))
(0.08732414245605469, 'torch.float32', (30522, 768))
(0.08732414245605469, 'torch.float32', (30522, 768))
(0.043662071228027344, 'torch.float32', (30522, 384))
(0.025634765625, 'torch.float16', (8960, 1536))
(0.025634765625, 'torch.float16', (8960, 1536))
(0.025634765625, 'torch.float16', (8960, 1536))
(0.025634765625, 'torch.float16', (8960, 1536))
(0.025634765625, 'torch.float16', (8960, 1536))


c:\Projects\DocuMind\ml\.venv\Lib\site-packages\torch\__init__.py:1538: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


In [1]:
import sys
import torch
import numpy as np
import pandas as pd
from pathlib import Path

print("DocuMind Agentic Orchestration")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory used:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

DocuMind Agentic Orchestration
Python: 3.12.10
PyTorch: 2.14.0+cu130
CUDA: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU memory used: 0.0 GB


In [2]:
import bm25s

processed_dir = Path("../processed")

full_chunks_df = pd.read_parquet(
    processed_dir / "full_text_chunks.parquet"
)

full_embeddings = np.load(
    processed_dir / "full_text_embeddings.npy",
    mmap_mode="r"
)

full_bm25s = bm25s.BM25.load(
    str(processed_dir / "full_text_bm25s"),
    load_corpus=False
)

print("Retrieval data loaded")
print("Chunks:", full_chunks_df.shape)
print("Embeddings:", full_embeddings.shape)
print("BM25:", type(full_bm25s).__name__)

Retrieval data loaded
Chunks: (223234, 6)
Embeddings: (223234, 384)
BM25: BM25


In [3]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device="cuda"
)

print("BGE loaded")
print("Device:", embedding_model.device)
print("Dimension:", embedding_model.get_embedding_dimension())
print(
    "GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE loaded
Device: cuda:0
Dimension: 384
GPU memory: 0.12 GB


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

qwen_name = "Qwen/Qwen2.5-1.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(
    qwen_name
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_name,
    torch_dtype=torch.float16
).to("cuda")

qwen_model.eval()

print("Qwen loaded")
print("Device:", next(qwen_model.parameters()).device)
print(
    "GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen loaded
Device: cuda:0
GPU memory: 3.0 GB


In [5]:
STATE = {
    "full_chunks_df": full_chunks_df,
    "full_embeddings": full_embeddings,
    "full_bm25s": full_bm25s,
    "embedding_model": embedding_model,
    "tokenizer": qwen_tokenizer,
    "model": qwen_model,
    "qwen_tokenizer": qwen_tokenizer,
    "qwen_model": qwen_model,
    "embeddings": full_embeddings
}

print("RAG state created")
print(list(STATE.keys()))

RAG state created
['full_chunks_df', 'full_embeddings', 'full_bm25s', 'embedding_model', 'tokenizer', 'model', 'qwen_tokenizer', 'qwen_model', 'embeddings']


In [6]:
# ============================================================
# Cell 6: Hybrid Search Tool
# ============================================================

def detect_document_type(question):
    q = question.lower()

    if "invoice" in q:
        return "Invoice"
    if "contract" in q:
        return "Contract"
    if "purchase order" in q or "purchase_order" in q or " po " in f" {q} ":
        return "Purchase Order"
    if "email" in q:
        return "Email"
    if "report" in q:
        return "Report"

    return None


def search_tool(
    question,
    top_k=5,
    candidate_k=30,
    label_filter=None
):
    # --------------------------------------------------------
    # BM25 retrieval
    # --------------------------------------------------------
    query_tokens = bm25s.tokenize([question])

    bm25_results, bm25_scores = full_bm25s.retrieve(
        query_tokens,
        k=candidate_k
    )

    bm25_indices = bm25_results[0]
    bm25_scores = bm25_scores[0]

    # --------------------------------------------------------
    # Semantic retrieval
    # --------------------------------------------------------
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    semantic_scores = np.dot(
        full_embeddings,
        query_embedding
    )

    semantic_indices = np.argsort(
        semantic_scores
    )[::-1][:candidate_k]

    # --------------------------------------------------------
    # Combine candidates
    # --------------------------------------------------------
    candidate_indices = set(
        [int(x) for x in bm25_indices]
        + [int(x) for x in semantic_indices]
    )

    results = []

    for idx in candidate_indices:

        row = full_chunks_df.iloc[idx]

        if (
            label_filter is not None
            and row["label"] != label_filter
        ):
            continue

        semantic_score = float(
            semantic_scores[idx]
        )

        # Find BM25 score if this document was retrieved by BM25
        bm25_score = 0.0

        for j in range(len(bm25_indices)):
            if int(bm25_indices[j]) == idx:
                bm25_score = float(bm25_scores[j])
                break

        # Simple normalized combination
        combined_score = (
            0.6 * semantic_score
            + 0.4 * (
                bm25_score /
                (1.0 + abs(bm25_score))
            )
        )

        results.append({
            "document_id": row["document_id"],
            "label": row["label"],
            "source": row["source"],
            "chunk_index": int(idx),
            "text": row["text"],
            "semantic_score": semantic_score,
            "bm25_score": bm25_score,
            "score": combined_score
        })

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return {
        "status": "success",
        "query": question,
        "count": len(results[:top_k]),
        "results": results[:top_k]
    }


print("Hybrid search tool created")

Hybrid search tool created


In [7]:
# ============================================================
# Cell 7: Document Classification Model
# ============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification

classifier_path = "../models/distilbert_doc_classifier/final"

classifier_tokenizer = AutoTokenizer.from_pretrained(
    classifier_path
)

classifier_model = AutoModelForSequenceClassification.from_pretrained(
    classifier_path,
    torch_dtype=torch.float16
).to("cuda")

classifier_model.eval()

print("Classification model loaded")
print("Device:", next(classifier_model.parameters()).device)
print("Labels:", classifier_model.config.id2label)
print(
    "GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Classification model loaded
Device: cuda:0
Labels: {0: 'Contract', 1: 'Email', 2: 'Invoice', 3: 'Purchase Order', 4: 'Report'}
GPU memory: 3.14 GB


In [8]:
# ============================================================
# Cell 8: Classification Tool
# ============================================================

import torch
import re

# Load the full corpus for exact document lookup
full_corpus_df = pd.read_parquet(
    processed_dir / "full_corpus.parquet"
)


def extract_document_id(question):
    match = re.search(
        r'\b(invoice|contract|email|report|purchase[_ ]?order)[_-]?(\d{3,5})\b',
        question,
        re.IGNORECASE
    )

    if not match:
        return None

    doc_type = match.group(1).lower()
    number = match.group(2)

    prefix_map = {
        "invoice": "invoice",
        "contract": "contract",
        "email": "email",
        "report": "report",
        "purchase order": "purchase_order",
        "purchase_order": "purchase_order"
    }

    return f"{prefix_map[doc_type]}_{number}"


def agent_classification_tool(question):

    document_id = extract_document_id(question)

    if document_id is None:
        return {
            "status": "error",
            "message": "Could not identify a document ID."
        }

    matches = full_corpus_df[
        full_corpus_df["document_id"].astype(str).str.lower()
        == document_id.lower()
    ]

    if len(matches) == 0:
        return {
            "status": "error",
            "message": f"Document {document_id} not found."
        }

    text = str(matches.iloc[0]["text"])

    # --------------------------------------------------------
    # Tokenize long document into overlapping chunks
    # --------------------------------------------------------
    encoded = classifier_tokenizer(
        text,
        truncation=True,
        max_length=512,
        stride=128,
        return_overflowing_tokens=True,
        padding="max_length",
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to("cuda")
    attention_mask = encoded["attention_mask"].to("cuda")

    # --------------------------------------------------------
    # Run classifier
    # --------------------------------------------------------
    with torch.no_grad():

        outputs = classifier_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        mean_logits = logits.mean(dim=0)

        probabilities = torch.softmax(
            mean_logits,
            dim=-1
        )

        predicted_id = torch.argmax(
            probabilities
        ).item()

        confidence = probabilities[
            predicted_id
        ].item()

    predicted_class = classifier_model.config.id2label[
        predicted_id
    ]

    probability_dict = {}

    for i in range(len(probabilities)):
        label = classifier_model.config.id2label[i]
        probability_dict[label] = round(
            probabilities[i].item(),
            4
        )

    return {
        "status": "success",
        "question": question,
        "document_id": document_id,
        "predicted_class": predicted_class,
        "confidence": round(confidence, 4),
        "num_chunks": int(input_ids.shape[0]),
        "probabilities": probability_dict,
        "method": "distilbert_document_classifier"
    }


print("Classification tool created")

Classification tool created


In [9]:
# ============================================================
# Cell 9: Metadata Extraction Tool
# ============================================================

def agent_metadata_tool(question):

    document_id = extract_document_id(question)

    if document_id is None:
        return {
            "status": "error",
            "message": "Could not identify a document ID."
        }

    # This metadata tool currently supports invoices
    if not document_id.startswith("invoice_"):
        return {
            "status": "error",
            "message": "Metadata extraction is currently implemented for invoices."
        }

    matches = full_corpus_df[
        full_corpus_df["document_id"].astype(str).str.lower()
        == document_id.lower()
    ]

    if len(matches) == 0:
        return {
            "status": "error",
            "message": f"Document {document_id} not found."
        }

    text = str(matches.iloc[0]["text"])

    q = question.lower()

    # --------------------------------------------------------
    # Determine requested field
    # --------------------------------------------------------
    if any(
        phrase in q
        for phrase in [
            "total amount",
            "gross amount",
            "total",
            "amount_total_gross"
        ]
    ):
        field = "amount_total_gross"

    elif any(
        phrase in q
        for phrase in [
            "amount due",
            "due amount",
            "balance due"
        ]
    ):
        field = "amount_due"

    elif any(
        phrase in q
        for phrase in [
            "invoice date",
            "issue date",
            "date issue"
        ]
    ):
        field = "date_issue"

    else:
        return {
            "status": "error",
            "message": "Could not determine the requested metadata field."
        }

    # --------------------------------------------------------
    # Invoice Totals table extraction
    # --------------------------------------------------------
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    start = None

    for i in range(len(lines)):
        if lines[i].lower() == "invoice totals":
            start = i
            break

    if start is not None:

        section = lines[start:start + 20]

        values = []

        currency_pattern = re.compile(
            r'^\(?\$[\d,]+(?:\.\d{2})?\)?$'
        )

        for line in section:
            if currency_pattern.match(line):
                values.append(line)

        # Expected order:
        # Gross Amount
        # Agency Commission
        # Net Amount Due
        #
        # The first currency value is the gross amount.
        # The last currency value is the amount due.

        if field == "amount_total_gross" and len(values) >= 1:

            return {
                "status": "success",
                "question": question,
                "document_id": document_id,
                "field": field,
                "value": values[0],
                "method": "invoice_totals_table_v2",
                "evidence": "\n".join(section)
            }

        if field == "amount_due" and len(values) >= 2:

            return {
                "status": "success",
                "question": question,
                "document_id": document_id,
                "field": field,
                "value": values[-1],
                "method": "invoice_totals_table_v2",
                "evidence": "\n".join(section)
            }

    # --------------------------------------------------------
    # Fallback
    # --------------------------------------------------------
    return {
        "status": "error",
        "message": f"Could not extract {field} from {document_id}."
    }


print("Metadata tool created")

Metadata tool created


In [21]:
# ============================================================
# Cell 10: RAG / Question Answering Tool
# ============================================================

def agent_rag_tool(question, top_k=3):

    document_id = extract_document_id(question)

    # --------------------------------------------------------
    # 1. Retrieve relevant chunks
    # --------------------------------------------------------
    if document_id is not None:

        doc_rows = full_chunks_df[
            full_chunks_df["document_id"].astype(str).str.lower()
            == document_id.lower()
        ].copy()

        if len(doc_rows) == 0:
            return {
                "status": "error",
                "message": f"Document {document_id} not found."
            }

        # Query embedding
        query_embedding = embedding_model.encode(
            question,
            normalize_embeddings=True
        )

        # Use the exact rows belonging to this document
        row_positions = doc_rows.index.to_numpy()

        doc_embeddings = full_embeddings[row_positions]

        scores = np.dot(
            doc_embeddings,
            query_embedding
        )

        best_positions = np.argsort(scores)[::-1][:top_k]

        contexts = []

        for rank in range(len(best_positions)):

            local_position = int(best_positions[rank])
            global_position = int(row_positions[local_position])

            row = full_chunks_df.iloc[global_position]

            contexts.append({
                "text": str(row["text"]),
                "score": float(scores[local_position]),
                "source_chunk": f"{document_id}_{local_position}"
            })

    else:

        search_result = search_tool(
            question,
            top_k=top_k
        )

        if search_result["status"] != "success":
            return search_result

        contexts = []

        for r in search_result["results"]:
            contexts.append({
                "text": str(r["text"]),
                "score": float(r["score"]),
                "source_chunk": r["document_id"]
            })

    # --------------------------------------------------------
    # 2. Build compact evidence
    # --------------------------------------------------------
    evidence = ""

    for i in range(len(contexts)):
        evidence += (
            f"\nSOURCE {i + 1}:\n"
            f"{contexts[i]['text'][:3000]}\n"
        )

    # --------------------------------------------------------
    # 3. Ask Qwen to answer from evidence only
    # --------------------------------------------------------
    prompt = f"""
You are an enterprise document question-answering assistant.

Answer the user's question using ONLY the evidence provided.

Rules:
- Do not invent information.
- Do not copy raw OCR sequences.
- Ignore duplicated values and OCR formatting noise.
- For explanation questions, summarize the document clearly.
- Keep the answer concise: 2 to 4 sentences.
- Mention important amounts, quantities, parties, or purpose only when clearly supported.

Question:
{question}

Evidence:
{evidence}

Answer:
"""

    inputs = qwen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    inputs = {
        key: value.to(qwen_model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        output = qwen_model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id
        )

    generated_tokens = output[
        0,
        inputs["input_ids"].shape[1]:
    ]

    answer = qwen_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

# Remove incomplete trailing text
    if answer and not answer.endswith((".", "!", "?")):

        matches = list(re.finditer(r"[.!?]", answer))

        if len(matches) > 0:
            answer = answer[:matches[-1].end()].strip()

# Final cleanup
    answer = answer.rstrip()

    return {
        "status": "success",
        "question": question,
        "document_id": document_id,
        "answer": answer,
        "source_chunk": contexts[0]["source_chunk"],
        "retrieval_score": round(contexts[0]["score"], 4),
        "method": "qwen_grounded_rag"
    }


print("RAG tool created")

RAG tool created


In [11]:
# ============================================================
# Cell 11: Agent Planner
# ============================================================

def plan_tools(question):

    q = question.lower()

    selected_tools = []

    # --------------------------------------------------------
    # Classification intent
    # --------------------------------------------------------
    classification_intent = any(
        word in q
        for word in [
            "type of document",
            "document type",
            "classify",
            "classification",
            "what type"
        ]
    )

    if classification_intent:
        selected_tools.append("classification")

    # --------------------------------------------------------
    # Metadata intent
    # --------------------------------------------------------
    metadata_intent = any(
        word in q
        for word in [
            "vendor",
            "supplier",
            "customer",
            "invoice number",
            "invoice date",
            "issue date",
            "total amount",
            "gross amount",
            "amount due",
            "due amount",
            "balance due",
            "metadata",
            "extract"
        ]
    )

    if metadata_intent:
        selected_tools.append("metadata")

    # --------------------------------------------------------
    # Search intent
    # --------------------------------------------------------
    search_intent = (
        "which " in q
        or "find " in q
        or "search for " in q
        or "list " in q
        or "show me " in q
    )

    specific_search_intent = any(
        phrase in q
        for phrase in [
            "which contracts mention",
            "which invoices mention",
            "which reports mention",
            "which emails mention",
            "which documents mention",
            "find contracts",
            "find invoices",
            "find reports",
            "find emails",
            "find documents"
        ]
    )

    if search_intent or specific_search_intent:
        selected_tools.append("search")

    # --------------------------------------------------------
    # RAG / QA intent
    # --------------------------------------------------------
    rag_intent = any(
        word in q
        for word in [
            "what",
            "how",
            "why",
            "explain",
            "describe",
            "tell me",
            "calculate",
            "meaning",
            "define",
            "contains",
            "contain"
        ]
    )

    if rag_intent:
        selected_tools.append("rag_qa")

    # --------------------------------------------------------
    # Remove duplicates
    # --------------------------------------------------------
    selected_tools = list(dict.fromkeys(selected_tools))

    # --------------------------------------------------------
    # Default
    # --------------------------------------------------------
    if len(selected_tools) == 0:
        selected_tools.append("rag_qa")

    return selected_tools


print("Agent planner created")

Agent planner created


In [12]:
# ============================================================
# Cell 12: Agent Tool Registry
# ============================================================

tools = {
    "classification": agent_classification_tool,
    "metadata": agent_metadata_tool,
    "search": search_tool,
    "rag_qa": agent_rag_tool
}

print("Agent tools registered")
print("-" * 50)

for name in tools:
    print(f"{name:15} -> {tools[name].__name__}")

Agent tools registered
--------------------------------------------------
classification  -> agent_classification_tool
metadata        -> agent_metadata_tool
search          -> search_tool
rag_qa          -> agent_rag_tool


In [13]:
# ============================================================
# Cell 13: Final Answer Formatter
# ============================================================

def format_final_answer(question, results):

    parts = []

    for i in range(len(results)):

        result = results[i]

        if result.get("status") != "success":
            continue

        tool = result.get("tool")

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------
        if tool == "classification":

            parts.append(
                f"The document type is "
                f"{result.get('predicted_class')}."
            )

        # ----------------------------------------------------
        # Metadata
        # ----------------------------------------------------
        elif tool == "metadata":

            field = result.get("field")
            value = result.get("value")

            if field == "amount_total_gross":
                parts.append(
                    f"The total amount is {value}."
                )

            elif field == "amount_due":
                parts.append(
                    f"The amount due is {value}."
                )

            else:
                parts.append(
                    f"{field}: {value}."
                )

        # ----------------------------------------------------
        # Search
        # ----------------------------------------------------
        elif tool == "search":

            count = result.get(
                "count",
                len(result.get("results", []))
            )

            search_results = result.get(
                "results",
                []
            )

            ids = []

            for j in range(len(search_results)):
                ids.append(
                    search_results[j]["document_id"]
                )

            search_term = result.get(
                "search_term",
                "requested term"
            )

            if len(ids) > 0:
                parts.append(
                    f'Found {count} documents mentioning '
                    f'"{search_term}": '
                    + ", ".join(ids) + "."
                )
            else:
                parts.append(
                    f'Found {count} documents mentioning '
                    f'"{search_term}".'
                )

        # ----------------------------------------------------
        # RAG
        # ----------------------------------------------------
        elif tool == "rag_qa":

            answer = result.get("answer")

            if answer:
                parts.append(answer)

    if len(parts) == 0:
        return "I could not find a grounded answer."

    return "\n\n".join(parts)


print("Final answer formatter created")

Final answer formatter created


In [14]:
# ============================================================
# Cell 14: Agent Runner
# ============================================================

def run_agent(question):

    # --------------------------------------------------------
    # 1. Plan tools
    # --------------------------------------------------------
    selected_tools = plan_tools(question)

    print("Question:", question)

    if len(selected_tools) == 1:
        print("Route:", selected_tools[0])
    else:
        print(
            "Routes:",
            " + ".join(selected_tools)
        )

    # --------------------------------------------------------
    # 2. Execute tools
    # --------------------------------------------------------
    results = []

    for i in range(len(selected_tools)):

        tool_name = selected_tools[i]
        tool = tools[tool_name]

        result = tool(question)

        # Add tool name explicitly for formatter
        result["tool"] = tool_name

        results.append(result)

        print(f"\nTool {i + 1}: {tool_name}")
        print("Status:", result.get("status", "unknown"))

        # Classification
        if tool_name == "classification":

            if result.get("status") == "success":
                print(
                    "Document type:",
                    result.get("predicted_class")
                )
                print(
                    "Confidence:",
                    result.get("confidence")
                )
            else:
                print(
                    "Message:",
                    result.get("message")
                )

        # Metadata
        elif tool_name == "metadata":

            if result.get("status") == "success":
                print(
                    "Field:",
                    result.get("field")
                )
                print(
                    "Value:",
                    result.get("value")
                )
            else:
                print(
                    "Message:",
                    result.get("message")
                )

        # Search
        elif tool_name == "search":

            if result.get("status") == "success":

                print(
                    "Results:",
                    result.get("count")
                )

                search_results = result.get(
                    "results",
                    []
                )

                for j in range(
                    min(len(search_results), 5)
                ):
                    print(
                        f"{j + 1}.",
                        search_results[j]["document_id"]
                    )
            else:
                print(
                    "Message:",
                    result.get("message")
                )

        # RAG
        elif tool_name == "rag_qa":

            if result.get("status") == "success":
                print(
                    "Answer:",
                    result.get("answer")
                )
                print(
                    "Source:",
                    result.get("source_chunk")
                )
                print(
                    "Method:",
                    result.get("method")
                )
            else:
                print(
                    "Message:",
                    result.get("message")
                )

    # --------------------------------------------------------
    # 3. Format final answer
    # --------------------------------------------------------
    final_answer = format_final_answer(
        question,
        results
    )

    print("\n" + "=" * 60)
    print("FINAL ANSWER")
    print("=" * 60)
    print(final_answer)

    # --------------------------------------------------------
    # 4. Return structured agent result
    # --------------------------------------------------------
    return {
        "question": question,
        "tools_used": selected_tools,
        "results": results,
        "final_answer": final_answer
    }


print("Agent runner created")

Agent runner created


In [15]:
result = run_agent(
    "What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?"
)

Question: What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?
Routes: classification + metadata + rag_qa

Tool 1: classification
Status: success
Document type: Invoice
Confidence: 0.9995

Tool 2: metadata
Status: success
Field: amount_total_gross
Value: $325.00

Tool 3: rag_qa
Status: success
Answer: The invoice_0292 is a sales invoice for media services with a net amount due of $276.25. It includes 13 spots at $25 each, resulting in a gross amount of $325.00. The agency commission is $48.75, leaving a net amount due of $276.25. This
Source: invoice_0292_0
Method: qwen_grounded_rag

FINAL ANSWER
The document type is Invoice.

The total amount is $325.00.

The invoice_0292 is a sales invoice for media services with a net amount due of $276.25. It includes 13 spots at $25 each, resulting in a gross amount of $325.00. The agency commission is $48.75, leaving a net amount due of $276.25. This


In [17]:
result = run_agent(
    "What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?"
)

Question: What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?
Routes: classification + metadata + rag_qa

Tool 1: classification
Status: success
Document type: Invoice
Confidence: 0.9995

Tool 2: metadata
Status: success
Field: amount_total_gross
Value: $325.00

Tool 3: rag_qa
Status: success
Answer: The invoice_0292 is a sales invoice for media services with a net amount due of $276.25. It includes 13 spots at $25 each, resulting in a gross amount of $325.00. The agency commission is $48.75, leaving a net amount due of $276.25. This
Source: invoice_0292_0
Method: qwen_grounded_rag

FINAL ANSWER
The document type is Invoice.

The total amount is $325.00.

The invoice_0292 is a sales invoice for media services with a net amount due of $276.25. It includes 13 spots at $25 each, resulting in a gross amount of $325.00. The agency commission is $48.75, leaving a net amount due of $276.25. This


In [19]:
result = run_agent(
    "What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?"
)

Question: What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?
Routes: classification + metadata + rag_qa

Tool 1: classification
Status: success
Document type: Invoice
Confidence: 0.9995

Tool 2: metadata
Status: success
Field: amount_total_gross
Value: $325.00

Tool 3: rag_qa
Status: success
Answer: The invoice_0292 is a sales invoice for media services with a net amount due of $276.25. It includes 13 spots at $25 each, resulting in a gross amount of $325.00. The agency commission is $48.75, leaving a net amount due of $276.25. This
Source: invoice_0292_0
Method: qwen_grounded_rag

FINAL ANSWER
The document type is Invoice.

The total amount is $325.00.

The invoice_0292 is a sales invoice for media services with a net amount due of $276.25. It includes 13 spots at $25 each, resulting in a gross amount of $325.00. The agency commission is $48.75, leaving a net amount due of $276.25. This


In [20]:
import inspect

source = inspect.getsource(agent_rag_tool)

print(source[source.find("answer = qwen_tokenizer.decode"):
             source.find("return {", source.find("answer = qwen_tokenizer.decode"))])

answer = qwen_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

# Remove an incomplete trailing sentence
    if answer and not re.search(r"[.!?]$", answer):
        sentences = re.split(
            r"(?<=[.!?])\s+",
            answer
        )

    if len(sentences) > 1:
        answer = " ".join(sentences[:-1]).strip()

# Final cleanup
    answer = answer.rstrip()

    


In [22]:
result = run_agent(
    "What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?"
)

Question: What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?
Routes: classification + metadata + rag_qa

Tool 1: classification
Status: success
Document type: Invoice
Confidence: 0.9995

Tool 2: metadata
Status: success
Field: amount_total_gross
Value: $325.00

Tool 3: rag_qa
Status: success
Answer: The invoice_0292 is a sales invoice for media services with a net amount due of $276.25. It includes 13 spots at $25 each, resulting in a gross amount of $325.00. The agency commission is $48.75, leaving a net amount due of $276.25. This
Source: invoice_0292_0
Method: qwen_grounded_rag

FINAL ANSWER
The document type is Invoice.

The total amount is $325.00.

The invoice_0292 is a sales invoice for media services with a net amount due of $276.25. It includes 13 spots at $25 each, resulting in a gross amount of $325.00. The agency commission is $48.75, leaving a net amount due of $276.25. This


In [23]:
def clean_rag_tool(question, top_k=3):

    result = agent_rag_tool(question, top_k)

    if result.get("status") == "success":

        answer = result.get("answer", "").strip()

        # Remove incomplete text after the last complete sentence.
        if answer and not answer.endswith((".", "!", "?")):

            matches = list(
                re.finditer(r"[.!?]", answer)
            )

            if matches:
                answer = answer[:matches[-1].end()].strip()

        result["answer"] = answer
        result["method"] = "qwen_grounded_rag_clean"

    return result


# Replace the RAG tool used by the agent
tools["rag_qa"] = clean_rag_tool

print("Clean RAG wrapper installed")
print("tools['rag_qa'] ->", tools["rag_qa"].__name__)

Clean RAG wrapper installed
tools['rag_qa'] -> clean_rag_tool


In [24]:
result = run_agent(
    "What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?"
)

Question: What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?
Routes: classification + metadata + rag_qa

Tool 1: classification
Status: success
Document type: Invoice
Confidence: 0.9995

Tool 2: metadata
Status: success
Field: amount_total_gross
Value: $325.00

Tool 3: rag_qa
Status: success
Answer: The invoice_0292 is a sales invoice for media services with a net amount due of $276.25. It includes 13 spots at $25 each, resulting in a gross amount of $325.00. The agency commission is $48.75, leaving a net amount due of $276.25. This invoice was sent to Eagle Radio of Great Bend on November 2nd, 2020.
Source: invoice_0292_0
Method: qwen_grounded_rag_clean

FINAL ANSWER
The document type is Invoice.

The total amount is $325.00.

The invoice_0292 is a sales invoice for media services with a net amount due of $276.25. It includes 13 spots at $25 each, resulting in a gross amount of $325.00. The agency commission is $48.75, leaving a ne

In [25]:
result = run_agent(
    "What type of document is invoice_0292?"
)

Question: What type of document is invoice_0292?
Routes: classification + rag_qa

Tool 1: classification
Status: success
Document type: Invoice
Confidence: 0.9995

Tool 2: rag_qa
Status: success
Answer: The document is an invoice for a radio station advertisement with the agency client code "VOTEVETS KSSEN R60" and total spots amounting to 13. The net amount due is $276.25. The terms are Day Date Time, with specific dates noted for each day of the week. The remittance address is Eagle Radio of Great Bend, Kansas. The product being advertised includes multiple instances of the same item (VOTEVETS KSSEN R60) totaling 60 units across all days.
Source: invoice_0292_0
Method: qwen_grounded_rag_clean

FINAL ANSWER
The document type is Invoice.

The document is an invoice for a radio station advertisement with the agency client code "VOTEVETS KSSEN R60" and total spots amounting to 13. The net amount due is $276.25. The terms are Day Date Time, with specific dates noted for each day of the we

In [26]:
# ============================================================
# Cell 11: Improved Agent Planner
# ============================================================

def plan_tools(question):

    q = question.lower().strip()

    selected_tools = []

    # --------------------------------------------------------
    # 1. Classification intent
    # --------------------------------------------------------
    classification_intent = any(
        phrase in q
        for phrase in [
            "type of document",
            "document type",
            "classify",
            "classification",
            "what type"
        ]
    )

    if classification_intent:
        selected_tools.append("classification")

    # --------------------------------------------------------
    # 2. Metadata intent
    # --------------------------------------------------------
    metadata_intent = any(
        phrase in q
        for phrase in [
            "vendor",
            "supplier",
            "customer",
            "invoice number",
            "invoice date",
            "issue date",
            "total amount",
            "gross amount",
            "amount due",
            "due amount",
            "balance due",
            "metadata",
            "extract"
        ]
    )

    if metadata_intent:
        selected_tools.append("metadata")

    # --------------------------------------------------------
    # 3. Search intent
    # --------------------------------------------------------
    search_intent = any(
        phrase in q
        for phrase in [
            "which contracts mention",
            "which invoices mention",
            "which reports mention",
            "which emails mention",
            "which documents mention",
            "find contracts",
            "find invoices",
            "find reports",
            "find emails",
            "find documents",
            "search for",
            "list documents",
            "show me documents"
        ]
    )

    if search_intent:
        selected_tools.append("search")

    # --------------------------------------------------------
    # 4. RAG / explanation intent
    # --------------------------------------------------------
    rag_intent = any(
        phrase in q
        for phrase in [
            "explain",
            "describe",
            "why",
            "how is",
            "how does",
            "how do",
            "calculate",
            "meaning",
            "define",
            "contains",
            "contain",
            "what is this document about",
            "what is this invoice about",
            "what is this contract about"
        ]
    )

    # Generic "what is" questions can use RAG,
    # but metadata/classification questions should not.
    generic_what_intent = (
        "what is " in q
        and not classification_intent
        and not metadata_intent
    )

    if rag_intent or generic_what_intent:
        selected_tools.append("rag_qa")

    # --------------------------------------------------------
    # Remove duplicates while preserving order
    # --------------------------------------------------------
    selected_tools = list(
        dict.fromkeys(selected_tools)
    )

    # --------------------------------------------------------
    # Default
    # --------------------------------------------------------
    if len(selected_tools) == 0:
        selected_tools.append("rag_qa")

    return selected_tools


print("Improved planner created")

Improved planner created


In [27]:
tests = [
    "What type of document is invoice_0292?",
    "What is the total amount of invoice_0292?",
    "Explain what this invoice contains.",
    "How is the termination fee calculated in contract_0266?",
    "Which contracts mention termination fees?"
]

for i in range(len(tests)):
    print("\nQuestion:", tests[i])
    print("Route:", " + ".join(plan_tools(tests[i])))


Question: What type of document is invoice_0292?
Route: classification

Question: What is the total amount of invoice_0292?
Route: metadata

Question: Explain what this invoice contains.
Route: rag_qa

Question: How is the termination fee calculated in contract_0266?
Route: rag_qa

Question: Which contracts mention termination fees?
Route: search


In [28]:
result = run_agent(
    "What type of document is invoice_0292?"
)

Question: What type of document is invoice_0292?
Route: classification

Tool 1: classification
Status: success
Document type: Invoice
Confidence: 0.9995

FINAL ANSWER
The document type is Invoice.


In [29]:
result = run_agent(
    "What is the total amount of invoice_0292?"
)

Question: What is the total amount of invoice_0292?
Route: metadata

Tool 1: metadata
Status: success
Field: amount_total_gross
Value: $325.00

FINAL ANSWER
The total amount is $325.00.


In [30]:
result = run_agent(
    "Explain what invoice_0292 contains."
)

Question: Explain what invoice_0292 contains.
Route: rag_qa

Tool 1: rag_qa
Status: success
Answer: Invoice_0292 is a sales invoice for advertising services with a total net amount due of $276.25. It includes multiple estimates for different products (VOTEVETS KSSEN R60) totaling 13 spots at $25 each. The invoice details specific dates, times, and remittance addresses for payments made on various days. The agency client code is ISCI, and the buyer name is Eagle Radio of Great Bend. Terms are Day Date Time, indicating payment deadlines based on business hours.
Source: invoice_0292_0
Method: qwen_grounded_rag_clean

FINAL ANSWER
Invoice_0292 is a sales invoice for advertising services with a total net amount due of $276.25. It includes multiple estimates for different products (VOTEVETS KSSEN R60) totaling 13 spots at $25 each. The invoice details specific dates, times, and remittance addresses for payments made on various days. The agency client code is ISCI, and the buyer name is Eagle

In [31]:
result = run_agent(
    "Which contracts mention termination fees?"
)

Question: Which contracts mention termination fees?
Route: search


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Tool 1: search
Status: success
Results: 5
1. contract_0428
2. report_0367
3. contract_0112
4. contract_0375
5. report_1673

FINAL ANSWER
Found 5 documents mentioning "requested term": contract_0428, report_0367, contract_0112, contract_0375, report_1673.


In [32]:
# ============================================================
# Cell 21: Improved Search Tool
# ============================================================

def search_tool(
    question,
    top_k=5,
    candidate_k=30,
    label_filter=None
):

    q = question.lower().strip()

    # --------------------------------------------------------
    # Detect enumeration questions
    # Example:
    # "Which contracts mention termination fees?"
    # --------------------------------------------------------
    match = re.search(
        r"which\s+(contracts|invoices|reports|emails|documents)"
        r"\s+mention\s+(.+?)(?:\?|$)",
        q
    )

    if match:

        requested_type = match.group(1)
        search_term = match.group(2).strip()

        type_map = {
            "contracts": "Contract",
            "invoices": "Invoice",
            "reports": "Report",
            "emails": "Email",
            "documents": None
        }

        requested_label = type_map[requested_type]

        # Remove surrounding quotes
        search_term = search_term.strip('"\' ')

        # Handle simple singular/plural variations
        term_variants = [search_term]

        if search_term.endswith("s"):
            term_variants.append(search_term[:-1])
        else:
            term_variants.append(search_term + "s")

        # ----------------------------------------------------
        # Search full documents, not chunks
        # ----------------------------------------------------
        search_df = full_corpus_df

        if requested_label is not None:
            search_df = search_df[
                search_df["label"] == requested_label
            ]

        matched_ids = []

        for i in range(len(search_df)):

            text = str(
                search_df.iloc[i]["text"]
            ).lower()

            found = False

            for variant in term_variants:
                if variant in text:
                    found = True
                    break

            if found:
                matched_ids.append(
                    search_df.iloc[i]["document_id"]
                )

        # ----------------------------------------------------
        # Return exhaustive document-level results
        # ----------------------------------------------------
        results = []

        for i in range(len(matched_ids)):

            results.append({
                "document_id": matched_ids[i],
                "label": requested_label,
                "search_term": search_term
            })

        return {
            "status": "success",
            "query": question,
            "search_term": search_term,
            "count": len(results),
            "results": results,
            "method": "exact_document_enumeration"
        }

    # --------------------------------------------------------
    # Normal hybrid search
    # --------------------------------------------------------

    query_tokens = bm25s.tokenize([question])

    bm25_results, bm25_scores = full_bm25s.retrieve(
        query_tokens,
        k=candidate_k
    )

    bm25_indices = bm25_results[0]
    bm25_scores = bm25_scores[0]

    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    semantic_scores = np.dot(
        full_embeddings,
        query_embedding
    )

    semantic_indices = np.argsort(
        semantic_scores
    )[::-1][:candidate_k]

    candidate_indices = set(
        [int(x) for x in bm25_indices]
        + [int(x) for x in semantic_indices]
    )

    results = []

    for idx in candidate_indices:

        row = full_chunks_df.iloc[idx]

        if (
            label_filter is not None
            and row["label"] != label_filter
        ):
            continue

        semantic_score = float(
            semantic_scores[idx]
        )

        bm25_score = 0.0

        for j in range(len(bm25_indices)):
            if int(bm25_indices[j]) == idx:
                bm25_score = float(bm25_scores[j])
                break

        combined_score = (
            0.6 * semantic_score
            + 0.4 * (
                bm25_score /
                (1.0 + abs(bm25_score))
            )
        )

        results.append({
            "document_id": row["document_id"],
            "label": row["label"],
            "source": row["source"],
            "chunk_index": int(idx),
            "text": row["text"],
            "semantic_score": semantic_score,
            "bm25_score": bm25_score,
            "score": combined_score
        })

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return {
        "status": "success",
        "query": question,
        "count": len(results[:top_k]),
        "results": results[:top_k],
        "method": "hybrid_search"
    }


# Make the agent use the new function
tools["search"] = search_tool

print("Improved search tool installed")

Improved search tool installed


In [33]:
run_agent("Which contracts mention termination fees?")

Question: Which contracts mention termination fees?
Route: search

Tool 1: search
Status: success
Results: 18
1. contract_0086
2. contract_0135
3. contract_0149
4. contract_0175
5. contract_0185

FINAL ANSWER
Found 18 documents mentioning "termination fees": contract_0086, contract_0135, contract_0149, contract_0175, contract_0185, contract_0256, contract_0266, contract_0299, contract_0302, contract_0329, contract_0348, contract_0385, contract_0400, contract_0408, contract_0442, contract_0448, contract_0479, contract_0481.


{'question': 'Which contracts mention termination fees?',
 'tools_used': ['search'],
 'results': [{'status': 'success',
   'query': 'Which contracts mention termination fees?',
   'search_term': 'termination fees',
   'count': 18,
   'results': [{'document_id': 'contract_0086',
     'label': 'Contract',
     'search_term': 'termination fees'},
    {'document_id': 'contract_0135',
     'label': 'Contract',
     'search_term': 'termination fees'},
    {'document_id': 'contract_0149',
     'label': 'Contract',
     'search_term': 'termination fees'},
    {'document_id': 'contract_0175',
     'label': 'Contract',
     'search_term': 'termination fees'},
    {'document_id': 'contract_0185',
     'label': 'Contract',
     'search_term': 'termination fees'},
    {'document_id': 'contract_0256',
     'label': 'Contract',
     'search_term': 'termination fees'},
    {'document_id': 'contract_0266',
     'label': 'Contract',
     'search_term': 'termination fees'},
    {'document_id': 'contract_

In [34]:
result = run_agent(
    "What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?"
)

Question: What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?
Routes: classification + metadata + rag_qa

Tool 1: classification
Status: success
Document type: Invoice
Confidence: 0.9995

Tool 2: metadata
Status: success
Field: amount_total_gross
Value: $325.00

Tool 3: rag_qa
Status: success
Answer: The invoice_0292 is a sales invoice for media services with a net amount due of $276.25. It includes 13 spots at $25 each, resulting in a gross amount of $325.00. The agency commission is $48.75, leaving a net amount due of $276.25. This invoice was sent to Eagle Radio of Great Bend on November 2nd, 2020.
Source: invoice_0292_0
Method: qwen_grounded_rag_clean

FINAL ANSWER
The document type is Invoice.

The total amount is $325.00.

The invoice_0292 is a sales invoice for media services with a net amount due of $276.25. It includes 13 spots at $25 each, resulting in a gross amount of $325.00. The agency commission is $48.75, leaving a ne

In [35]:
# ============================================================
# Cell 23: Agent Demo Suite
# ============================================================

demo_questions = [
    "What type of document is invoice_0292?",
    "What is the total amount of invoice_0292?",
    "Explain what invoice_0292 contains.",
    "How is the termination fee calculated in contract_0266?",
    "Which contracts mention termination fees?",
    "What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?"
]

for i in range(len(demo_questions)):

    print("\n" + "=" * 70)
    print(f"DEMO {i + 1}")
    print("=" * 70)

    result = run_agent(demo_questions[i])

    print("\nTools used:", result["tools_used"])


DEMO 1
Question: What type of document is invoice_0292?
Route: classification

Tool 1: classification
Status: success
Document type: Invoice
Confidence: 0.9995

FINAL ANSWER
The document type is Invoice.

Tools used: ['classification']

DEMO 2
Question: What is the total amount of invoice_0292?
Route: metadata

Tool 1: metadata
Status: success
Field: amount_total_gross
Value: $325.00

FINAL ANSWER
The total amount is $325.00.

Tools used: ['metadata']

DEMO 3
Question: Explain what invoice_0292 contains.
Route: rag_qa

Tool 1: rag_qa
Status: success
Answer: Invoice_0292 is a sales invoice for advertising services with a total net amount due of $276.25. It includes multiple estimates for different products (VOTEVETS KSSEN R60) totaling 13 spots at $25 each. The invoice details specific dates, times, and remittance addresses for payments made on various days. The agency client code is ISCI, and the buyer name is Eagle Radio of Great Bend. Terms are Day Date Time, indicating payment dead

In [36]:
# ============================================================
# Cell 24: Automated Agent Validation
# ============================================================

validation_cases = [
    {
        "question": "What type of document is invoice_0292?",
        "expected_tools": ["classification"]
    },
    {
        "question": "What is the total amount of invoice_0292?",
        "expected_tools": ["metadata"]
    },
    {
        "question": "Explain what invoice_0292 contains.",
        "expected_tools": ["rag_qa"]
    },
    {
        "question": "How is the termination fee calculated in contract_0266?",
        "expected_tools": ["rag_qa"]
    },
    {
        "question": "Which contracts mention termination fees?",
        "expected_tools": ["search"]
    },
    {
        "question": (
            "What type of document is invoice_0292, "
            "what is its total amount, and explain what this invoice contains?"
        ),
        "expected_tools": [
            "classification",
            "metadata",
            "rag_qa"
        ]
    }
]

passed = 0

print("=" * 70)
print("AGENT VALIDATION")
print("=" * 70)

for i in range(len(validation_cases)):

    case = validation_cases[i]

    actual_tools = plan_tools(
        case["question"]
    )

    expected_tools = case["expected_tools"]

    is_correct = (
        actual_tools == expected_tools
    )

    if is_correct:
        passed += 1

    print(f"\nTest {i + 1}")
    print("Question:", case["question"])
    print("Expected:", expected_tools)
    print("Actual:  ", actual_tools)
    print("Result:  ", "PASS" if is_correct else "FAIL")

print("\n" + "=" * 70)
print(
    f"Planner accuracy: "
    f"{passed}/{len(validation_cases)} "
    f"({passed / len(validation_cases) * 100:.1f}%)"
)
print("=" * 70)

AGENT VALIDATION

Test 1
Question: What type of document is invoice_0292?
Expected: ['classification']
Actual:   ['classification']
Result:   PASS

Test 2
Question: What is the total amount of invoice_0292?
Expected: ['metadata']
Actual:   ['metadata']
Result:   PASS

Test 3
Question: Explain what invoice_0292 contains.
Expected: ['rag_qa']
Actual:   ['rag_qa']
Result:   PASS

Test 4
Question: How is the termination fee calculated in contract_0266?
Expected: ['rag_qa']
Actual:   ['rag_qa']
Result:   PASS

Test 5
Question: Which contracts mention termination fees?
Expected: ['search']
Actual:   ['search']
Result:   PASS

Test 6
Question: What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?
Expected: ['classification', 'metadata', 'rag_qa']
Actual:   ['classification', 'metadata', 'rag_qa']
Result:   PASS

Planner accuracy: 6/6 (100.0%)


In [37]:
# ============================================================
# Cell 25: Agent Execution Trace
# ============================================================

def show_agent_trace(question):

    selected_tools = plan_tools(question)

    print("=" * 70)
    print("DOCUMIND AGENT EXECUTION TRACE")
    print("=" * 70)

    print("\nUSER QUESTION")
    print(question)

    print("\nSTEP 1 — INTENT ANALYSIS")
    print("Detected tools:", selected_tools)

    print("\nSTEP 2 — TOOL SELECTION")

    for i in range(len(selected_tools)):
        print(
            f"  {i + 1}. {selected_tools[i]}"
        )

    print("\nSTEP 3 — EXECUTION ORDER")

    for i in range(len(selected_tools)):
        print(
            f"  Agent → {selected_tools[i]}"
        )

    print("\nSTEP 4 — FINAL ANSWER")
    print("Answer generated after tool execution.")

    print("\n" + "=" * 70)


show_agent_trace(
    "What type of document is invoice_0292, "
    "what is its total amount, and explain what this invoice contains?"
)

DOCUMIND AGENT EXECUTION TRACE

USER QUESTION
What type of document is invoice_0292, what is its total amount, and explain what this invoice contains?

STEP 1 — INTENT ANALYSIS
Detected tools: ['classification', 'metadata', 'rag_qa']

STEP 2 — TOOL SELECTION
  1. classification
  2. metadata
  3. rag_qa

STEP 3 — EXECUTION ORDER
  Agent → classification
  Agent → metadata
  Agent → rag_qa

STEP 4 — FINAL ANSWER
Answer generated after tool execution.



In [38]:
# ============================================================
# Cell 26: DocuMind Agent Summary
# ============================================================

summary = {
    "document_classifier": "DistilBERT",
    "document_classes": [
        "Contract",
        "Email",
        "Invoice",
        "Purchase Order",
        "Report"
    ],
    "metadata_extraction": "Invoice field extraction",
    "semantic_search": "BGE-small-en-v1.5",
    "keyword_search": "BM25",
    "question_answering": "Qwen2.5-1.5B-Instruct",
    "agent_planner": "Rule-based intent planner",
    "multi_tool_orchestration": True,
    "planner_validation": "6/6 tests passed",
    "document_corpus": 6638,
    "retrieval_chunks": 223234
}

print("=" * 70)
print("DOCUMIND | AGENTIC ORCHESTRATION SUMMARY")
print("=" * 70)

for key, value in summary.items():
    print(f"{key:30} : {value}")

print("=" * 70)
print("Notebook 06 completed")
print("=" * 70)

DOCUMIND | AGENTIC ORCHESTRATION SUMMARY
document_classifier            : DistilBERT
document_classes               : ['Contract', 'Email', 'Invoice', 'Purchase Order', 'Report']
metadata_extraction            : Invoice field extraction
semantic_search                : BGE-small-en-v1.5
keyword_search                 : BM25
question_answering             : Qwen2.5-1.5B-Instruct
agent_planner                  : Rule-based intent planner
multi_tool_orchestration       : True
planner_validation             : 6/6 tests passed
document_corpus                : 6638
retrieval_chunks               : 223234
Notebook 06 completed
